# CLaP state-occurrence object study

Generated from `clap_researcher_input.ipynb`. CLaP remains the state detector.
The public `featuregraph.from_state_sequence` adapter preserves its output as
observation-, event-, object-, and relation-level representations without
changing the inferred labels.

## Environment setup

This study uses CLaP as an external detector. Its dependencies are deliberately
optional rather than part of FeatureGraph's core installation. From the
repository root, run:

```python
%pip install -e ".[clap-study]"
```

Then restart the notebook kernel before running the study. The equivalent
requirements-file installation is `%pip install -r notebooks/clap_requirements.txt`.

In [ ]:
from importlib.metadata import version

import featuregraph as fg
from featuregraph.behaviors.feature_object import ObjectStatus
try:
    from claspy.data_loader import load_tssb_dataset
    from claspy.state_detection import AgglomerativeCLaPDetection
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "The CLaP study dependencies are not installed in this kernel. "
        "Run `%pip install -e '.[clap-study]'` from the repository root, "
        "restart the kernel, and run the notebook again."
    ) from exc
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_mutual_info_score, adjusted_rand_score

DATASET_NAME = "Crop"
EXPECTED_CLASPY_VERSION = "0.2.8"


def load_crop():
    row = load_tssb_dataset(names=(DATASET_NAME,)).iloc[0]
    dataset, window_size, true_cps, reference_states, signal = row
    return {
        "dataset": dataset,
        "window_size": int(window_size),
        "true_change_points": np.asarray(true_cps, dtype=int),
        "reference_states": np.asarray(reference_states),
        "signal": np.asarray(signal, dtype=float),
    }


def run_clap(signal):
    detector = AgglomerativeCLaPDetection()
    clap_states = np.asarray(detector.fit_predict(signal))
    sparse_states, sparse_transitions = detector.predict(sparse=True)
    return detector, clap_states, set(sparse_states), set(sparse_transitions)


def compare_boundaries(predicted_change_points, true_change_points):
    if len(predicted_change_points) != len(true_change_points):
        raise ValueError(
            "Ordered boundary comparison requires equal counts; declare a matching rule first."
        )
    comparison = pd.DataFrame({
        "boundary_number": np.arange(1, len(true_change_points) + 1),
        "reference_index": true_change_points,
        "clap_index": predicted_change_points,
    })
    comparison["signed_error_samples"] = (
        comparison["clap_index"] - comparison["reference_index"]
    )
    comparison["absolute_error_samples"] = comparison[
        "signed_error_samples"
    ].abs()
    comparison["matching_rule"] = "ordered_equal_count"
    return comparison


def validate_study(source, clap_states, result, sparse_transitions):
    observations = result.observations
    objects = result.object_table()
    predicted_cps = observations.loc[
        observations["enter_state_occurrence"] & observations["sample_index"].gt(0),
        "sample_index",
    ].to_numpy()
    relation_pairs = set(map(tuple, result.relations[
        ["source_state", "target_state"]
    ].to_numpy()))
    checks = {
        "package_adapter_used": result.__class__.__module__.startswith("featuregraph."),
        "raw_signal_preserved": np.array_equal(
            observations["signal_raw"].to_numpy(), source["signal"]
        ),
        "one_label_per_observation": observations["state_label"].notna().all(),
        "one_state_per_occurrence": observations.groupby("occurrence_id")[
            "state_label"
        ].nunique().eq(1).all(),
        "occurrences_cover_source": int(objects["sample_count"].sum()) == len(observations),
        "occurrence_ids_consecutive": observations["occurrence_id"].drop_duplicates().tolist()
        == list(range(len(objects))),
        "state_sequence_reconstructed": np.array_equal(
            result.reconstruct_states(), clap_states
        ),
        "change_points_match_object_starts": np.array_equal(
            predicted_cps, objects["start_index"].iloc[1:].to_numpy()
        ),
        "one_relation_per_adjacency": len(result.relations) == max(0, len(objects) - 1),
        "relations_match_sparse_clap_graph": relation_pairs
        == set(map(tuple, sparse_transitions)),
        "edge_fragments_retained": objects["status"].eq(
            ObjectStatus.BOUNDARY_TRUNCATED.value
        ).sum() == 2,
    }
    report = pd.Series(checks, name="passed").rename_axis("check").to_frame()
    failed = report.index[~report["passed"]].tolist()
    if failed:
        raise AssertionError(f"CLaP object study validation failed: {failed}")
    return report


In [ ]:
source = load_crop()
detector, clap_states, sparse_states, sparse_transitions = run_clap(source["signal"])

# This is the software boundary under study: CLaP supplies labels and the
# FeatureGraph package materializes them as occurrences and relations.
object_result = fg.from_state_sequence(
    clap_states,
    signal=source["signal"],
    group_id="CLAP-CROP",
    dataset=source["dataset"],
    signal_name="Crop signal",
    detector=f"claspy.{type(detector).__name__}",
    software_version=fg.__version__,
)
observations = object_result.observations.copy()
observations["reference_state"] = source["reference_states"]
state_occurrences = object_result.object_table()
occurrence_relations = object_result.relations
predicted_change_points = observations.loc[
    observations["enter_state_occurrence"] & observations["sample_index"].gt(0),
    "sample_index",
].to_numpy()
boundary_comparison = compare_boundaries(
    predicted_change_points, source["true_change_points"]
)
validation_report = validate_study(
    source, clap_states, object_result, sparse_transitions
)

provenance = {
    "dataset": source["dataset"],
    "observations": len(observations),
    "window_size_from_benchmark": source["window_size"],
    "claspy_version": version("claspy"),
    "detector": type(detector).__name__,
    "detector_configuration": "defaults",
    "featuregraph_version": fg.__version__,
    "materializer": "featuregraph.from_state_sequence",
    "inferred_state_classes": sorted(map(int, sparse_states)),
    "sparse_transitions": sorted(map(lambda pair: tuple(map(int, pair)), sparse_transitions)),
}

summary = pd.Series({
    "observations": len(observations),
    "reference_change_points": len(source["true_change_points"]),
    "clap_change_points": len(predicted_change_points),
    "state_occurrences": len(state_occurrences),
    "complete_internal_occurrences": state_occurrences["status"].eq("complete").sum(),
    "boundary_fragments": state_occurrences["status"].eq("boundary_truncated").sum(),
    "adjacent_relations": len(occurrence_relations),
    "adjusted_rand_index": adjusted_rand_score(source["reference_states"], clap_states),
    "adjusted_mutual_information": adjusted_mutual_info_score(source["reference_states"], clap_states),
    "median_absolute_boundary_error_samples": boundary_comparison["absolute_error_samples"].median(),
    "maximum_absolute_boundary_error_samples": boundary_comparison["absolute_error_samples"].max(),
}, name="value").rename_axis("measure").to_frame()

display(pd.Series(provenance, name="value").rename_axis("field").to_frame())
display(validation_report)
display(summary)
display(boundary_comparison)
display(state_occurrences)
display(occurrence_relations)


## Interpretation limits

CLaP supplies the inferred state sequence. The public FeatureGraph package
adapter `featuregraph.from_state_sequence` does not alter or improve those
labels; it materializes each maximal run as a `FeatureObject`, retains the
first and final runs as boundary-truncated objects, and emits adjacency
relations and detector provenance. State-class integers are nominal identifiers
rather than crop meanings. Results from this one benchmark series establish a
tested software interface, not general interoperability.